# 🎮 아이작의 번제 아이템 파서

**Google Colab용 완전 자동화 도구**

## 📋 기능
1. HTML 파일 업로드
2. 아이템 정보 자동 파싱
3. Google Drive에 자동 저장

## 🚀 사용 방법
1. 아래 셀들을 순서대로 실행
2. HTML 파일 업로드
3. 아이템 타입 선택
4. 자동 완료!

In [ ]:
# ============================================================
# 1️⃣ 라이브러리 설치 및 임포트
# ============================================================

!pip install beautifulsoup4 -q

from google.colab import files, drive
from bs4 import BeautifulSoup
import json
import re
import requests
import time
import os
from pathlib import Path

print("✅ 라이브러리 설치 완료!")

In [ ]:
# ============================================================
# 2️⃣ Google Drive 연결
# ============================================================

print("📁 Google Drive 연결 중...")
drive.mount('/content/drive')
print("✅ Google Drive 연결 완료!")

# 저장 경로 설정
DRIVE_PATH = '/content/drive/MyDrive/isaac_items'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/images', exist_ok=True)

print(f"✅ 저장 경로 생성: {DRIVE_PATH}")

In [ ]:
# ============================================================
# 3️⃣ HTML 파일 업로드
# ============================================================

print("="*60)
print("📤 HTML 파일을 업로드하세요")
print("="*60)

uploaded = files.upload()

html_filename = list(uploaded.keys())[0]
print(f"\n✅ 업로드 완료: {html_filename}")

In [ ]:
# ============================================================
# 4️⃣ 파싱 함수 정의
# ============================================================

def parse_html_items(html_content, item_type='active'):
    """HTML에서 아이템 정보 파싱"""
    
    soup = BeautifulSoup(html_content, 'html.parser')
    tables = soup.find_all('table', class_='XrDIkehY')
    
    items = []
    item_id = 1
    
    print(f"\n찾은 테이블 수: {len(tables)}")
    
    for table in tables:
        item = {
            'id': item_id,
            'type': item_type,
            'name': '',
            'english_name': '',
            'icon': '🎯' if item_type == 'active' else '⭐' if item_type == 'passive' else '💎',
            'image_url': '',
            'description': '',
            'cooldown': '',
            'unlock_condition': '',
            'location': '',
            'grade': '',
            'effect': '',
            'game_id': ''
        }
        
        rows = table.find_all('tr')
        
        for row in rows:
            cells = row.find_all('td')
            
            if len(cells) == 1 and cells[0].get('colspan') == '2':
                title_text = cells[0].get_text(strip=True)
                name_match = re.match(r'^(.+?)\s*\((.+?)\)$', title_text)
                if name_match:
                    item['english_name'] = name_match.group(1).strip()
                    item['name'] = name_match.group(2).strip()
            
            elif len(cells) == 2:
                label = cells[0].get_text(strip=True)
                content_cell = cells[1]
                
                if label == '이미지':
                    imgs = content_cell.find_all('img', class_='udsARbbk')
                    for img in imgs:
                        if 'src' in img.attrs:
                            src = img['src']
                            if src.startswith('//'):
                                item['image_url'] = 'https:' + src
                                break
                
                elif label == 'ID':
                    game_id = content_cell.get_text(strip=True)
                    game_id = re.sub(r'\[.*?\]', '', game_id).strip()
                    item['game_id'] = game_id
                
                elif label == '습득 시 대사':
                    text = content_cell.get_text(strip=True)
                    korean_match = re.search(r'\(([^)]+)\)(?:\s*\[.*?\])?$', text)
                    if korean_match:
                        item['description'] = korean_match.group(1).strip()
                    else:
                        item['description'] = text
                
                elif label == '쿨타임':
                    cooldown = content_cell.get_text(strip=True)
                    cooldown_match = re.search(r'(\d+)칸', cooldown)
                    if cooldown_match:
                        item['cooldown'] = cooldown_match.group(1) + '칸'
                    elif '무제한' in cooldown:
                        item['cooldown'] = '무제한'
                
                elif label == '언락 조건':
                    unlock = content_cell.get_text(strip=True)
                    unlock = re.sub(r'\[.*?\]', '', unlock).strip()
                    item['unlock_condition'] = unlock if unlock and unlock != '-' else '없음'
                
                elif label == '등장 장소':
                    locations = []
                    for text_node in content_cell.stripped_strings:
                        if text_node and len(text_node) > 1:
                            locations.append(text_node)
                    item['location'] = ', '.join(locations[:5]) if locations else '없음'
                
                elif label == '등급':
                    grade = content_cell.get_text(strip=True)
                    grade_match = re.search(r'(\d+)등급', grade)
                    if grade_match:
                        item['grade'] = grade_match.group(1) + '등급'
                
                elif label == '설명':
                    lis = content_cell.find_all('li')
                    effect_parts = []
                    for li in lis:
                        for a in li.find_all('a', class_='KHXRqSq-'):
                            a.decompose()
                        text = li.get_text(strip=True)
                        if text:
                            effect_parts.append(text)
                    
                    if effect_parts:
                        item['effect'] = ' '.join(effect_parts[:2])
                        if len(item['effect']) > 250:
                            item['effect'] = item['effect'][:250] + '...'
        
        if item['name'] and item['image_url']:
            items.append(item)
            print(f"  ✓ {item['name']} (ID: {item['game_id']})")
            item_id += 1
    
    return items

print("✅ 파싱 함수 정의 완료!")

In [ ]:
# ============================================================
# 6️⃣ 아이템 타입 선택 & HTML 파싱
# ============================================================

print("\n아이템 타입을 선택하세요:")
print("1. 액티브 (active)")
print("2. 패시브 (passive)")
print("3. 장신구 (trinket)")

item_type_input = input("\n번호 입력 (1-3): ").strip()
item_type_map = {'1': 'active', '2': 'passive', '3': 'trinket'}
item_type = item_type_map.get(item_type_input, 'active')

print(f"\n✅ 선택: {item_type}")
print(f"\n{'='*60}")
print("🔍 HTML 파싱 중...")
print(f"{'='*60}")

# HTML 파싱
with open(html_filename, 'r', encoding='utf-8') as f:
    html_content = f.read()

items = parse_html_items(html_content, item_type)

print(f"\n✅ 총 {len(items)}개 아이템 파싱 완료!")

# 미리보기
print(f"\n{'='*60}")
print("처음 3개 미리보기:")
print(f"{'='*60}")
for item in items[:3]:
    icon = '🎯' if item['type'] == 'active' else '⭐' if item['type'] == 'passive' else '💎'
    print(f"\n{icon} {item['name']} ({item['english_name']})")
    print(f"   ID: {item['game_id']}")
    print(f"   {item['description']}")

In [ ]:
# ============================================================
# 8️⃣ 파일 저장 (JSON & JS)
# ============================================================

print(f"\n{'='*60}")
print("💾 파일 저장 중...")
print(f"{'='*60}\n")

base_filename = f"{item_type}_items"

# JSON 저장
json_path = f"{DRIVE_PATH}/{base_filename}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)
print(f"✅ JSON: {base_filename}.json")

# JavaScript 저장
js_path = f"{DRIVE_PATH}/{base_filename}.js"
js_content = f"const {item_type}ItemsData = " + json.dumps(items, ensure_ascii=False, indent=2) + ";"
with open(js_path, 'w', encoding='utf-8') as f:
    f.write(js_content)
print(f"✅ JS: {base_filename}.js")

print(f"\n{'='*60}")
print("✨ 완료!")
print(f"{'='*60}")
print(f"\n📊 결과 요약:")
print(f"  • 아이템: {len(items)}개")
print(f"\n📁 Google Drive 위치:")
print(f"  MyDrive/isaac_items/")
print(f"  ├── {base_filename}.json")
print(f"  ├── {base_filename}.js")
